# Ensemble Model Evaluation

Este notebook combina los 3 modelos entrenados (EfficientNetB3, ResNet50, VGG16) en un **ensemble** mediante **voting promediado**.

**Objetivo:** ≥95% accuracy en test set

**Estrategia:** Soft voting (promedio de probabilidades)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
from datetime import datetime

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score

print(f"TensorFlow version: {tf.__version__}")

np.random.seed(42)

In [ ]:
# Configuration
BASE_DIR = Path(r'C:\Users\hecto\OneDrive\Desktop\Ciencia de Datos')
DATA_DIR = BASE_DIR / 'data' / 'processed'
MODELS_DIR = BASE_DIR / 'ml_models'
ENSEMBLE_DIR = MODELS_DIR / 'ensemble'
ENSEMBLE_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

print(f"Models directory: {MODELS_DIR}")
print(f"Ensemble directory: {ENSEMBLE_DIR}")

## 1. Load Test Data

In [ ]:
test_datagen = ImageDataGenerator()

test_generator = test_datagen.flow_from_directory(
    DATA_DIR / 'test',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

class_names = list(test_generator.class_indices.keys())
print(f"Test samples: {test_generator.samples}")
print(f"Classes: {class_names}")

## 2. Load Individual Models

In [ ]:
print("Loading models...\n")

# Load EfficientNetB3
efficientnet_path = MODELS_DIR / 'efficientnet' / 'efficientnet_best.keras'
if efficientnet_path.exists():
    model_efficientnet = keras.models.load_model(efficientnet_path)
    print("✅ EfficientNetB3 loaded")
else:
    print("❌ EfficientNetB3 not found")
    model_efficientnet = None

# Load ResNet50
resnet_path = MODELS_DIR / 'resnet50' / 'resnet50_best.keras'
if resnet_path.exists():
    model_resnet = keras.models.load_model(resnet_path)
    print("✅ ResNet50 loaded")
else:
    print("❌ ResNet50 not found")
    model_resnet = None

# Load VGG16
vgg_path = MODELS_DIR / 'vgg16' / 'vgg16_best.keras'
if vgg_path.exists():
    model_vgg = keras.models.load_model(vgg_path)
    print("✅ VGG16 loaded")
else:
    print("❌ VGG16 not found")
    model_vgg = None

# Collect available models
models = []
model_names = []
if model_efficientnet:
    models.append(model_efficientnet)
    model_names.append('EfficientNetB3')
if model_resnet:
    models.append(model_resnet)
    model_names.append('ResNet50')
if model_vgg:
    models.append(model_vgg)
    model_names.append('VGG16')

print(f"\nTotal models loaded: {len(models)}")
print(f"Models: {', '.join(model_names)}")

## 3. Individual Model Performance

In [ ]:
print("="*80)
print("INDIVIDUAL MODEL PERFORMANCE")
print("="*80)

y_true = test_generator.classes
individual_predictions = []
individual_accuracies = {}

for model, name in zip(models, model_names):
    test_generator.reset()
    y_pred_probs = model.predict(test_generator, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)
    
    acc = accuracy_score(y_true, y_pred)
    individual_accuracies[name] = acc
    individual_predictions.append(y_pred_probs)
    
    print(f"\n{name}:")
    print(f"  Accuracy: {acc*100:.2f}%")

print("\n" + "="*80)

## 4. Ensemble Prediction (Soft Voting)

In [ ]:
print("="*80)
print("ENSEMBLE MODEL - SOFT VOTING")
print("="*80)

# Average predictions from all models
ensemble_probs = np.mean(individual_predictions, axis=0)
ensemble_pred = np.argmax(ensemble_probs, axis=1)

# Calculate metrics
ensemble_acc = accuracy_score(y_true, ensemble_pred)
ensemble_auc = roc_auc_score(test_generator.labels, ensemble_probs, multi_class='ovr')

print(f"\n📊 ENSEMBLE RESULTS:")
print(f"  Number of models: {len(models)}")
print(f"  Ensemble Accuracy: {ensemble_acc*100:.2f}%")
print(f"  Ensemble AUC: {ensemble_auc:.4f}")

print(f"\n📈 IMPROVEMENT:")
best_individual = max(individual_accuracies.values())
improvement = (ensemble_acc - best_individual) * 100
print(f"  Best individual: {best_individual*100:.2f}%")
print(f"  Ensemble: {ensemble_acc*100:.2f}%")
print(f"  Improvement: {improvement:+.2f}%")

if ensemble_acc >= 0.95:
    print(f"\n✅ TARGET ACHIEVED! {ensemble_acc*100:.2f}% ≥ 95%")
else:
    print(f"\n⚠️ Target not met: {ensemble_acc*100:.2f}% < 95%")

## 5. Detailed Classification Report

In [ ]:
print("\n" + "="*80)
print("ENSEMBLE CLASSIFICATION REPORT")
print("="*80)
print(classification_report(y_true, ensemble_pred, target_names=class_names, digits=4))

## 6. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, ensemble_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', 
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Count'})
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.title(f'Confusion Matrix - Ensemble ({ensemble_acc*100:.2f}% Accuracy)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(ENSEMBLE_DIR / 'ensemble_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Confusion matrix saved!")

## 7. Model Comparison

In [ ]:
# Compare all models
comparison_data = individual_accuracies.copy()
comparison_data['Ensemble'] = ensemble_acc

fig, ax = plt.subplots(figsize=(12, 6))

models_list = list(comparison_data.keys())
accuracies = [comparison_data[m]*100 for m in models_list]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#9467bd'][:len(models_list)]

bars = ax.bar(models_list, accuracies, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)

# Add value labels on bars
for bar, acc in zip(bars, accuracies):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{acc:.2f}%',
            ha='center', va='bottom', fontsize=12, fontweight='bold')

# Add target line
ax.axhline(y=95, color='red', linestyle='--', linewidth=2, label='Target (95%)', alpha=0.7)

ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_ylim([80, 100])
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(ENSEMBLE_DIR / 'model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Comparison chart saved!")

## 8. Per-Class Performance

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

precision, recall, f1, support = precision_recall_fscore_support(y_true, ensemble_pred)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metrics = [precision, recall, f1]
titles = ['Precision', 'Recall', 'F1-Score']
colors_map = ['#8dd3c7', '#fdb462', '#bebada']

for ax, metric, title, color in zip(axes, metrics, titles, colors_map):
    bars = ax.bar(class_names, metric, color=color, alpha=0.8, edgecolor='black')
    ax.set_ylabel(title, fontsize=12)
    ax.set_title(f'{title} by Class', fontsize=13, fontweight='bold')
    ax.set_ylim([0, 1.1])
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels
    for bar, val in zip(bars, metric):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                f'{val:.3f}', ha='center', fontsize=10)
    
    ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig(ENSEMBLE_DIR / 'per_class_metrics.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Per-class metrics chart saved!")

## 9. Save Ensemble Results

In [ ]:
# Save ensemble metrics
ensemble_metrics = {
    'ensemble_type': 'soft_voting',
    'num_models': len(models),
    'models': model_names,
    'test_accuracy': float(ensemble_acc),
    'test_auc': float(ensemble_auc),
    'individual_accuracies': {k: float(v) for k, v in individual_accuracies.items()},
    'best_individual_accuracy': float(best_individual),
    'improvement_over_best': float(improvement),
    'class_names': class_names,
    'per_class_precision': precision.tolist(),
    'per_class_recall': recall.tolist(),
    'per_class_f1': f1.tolist(),
    'per_class_support': support.tolist(),
    'evaluation_date': datetime.now().isoformat()
}

with open(ENSEMBLE_DIR / 'ensemble_metrics.json', 'w') as f:
    json.dump(ensemble_metrics, f, indent=2)

print(f"✅ Ensemble metrics saved to: {ENSEMBLE_DIR / 'ensemble_metrics.json'}")

# Save ensemble predictions for later use
np.save(ENSEMBLE_DIR / 'ensemble_predictions.npy', ensemble_probs)
np.save(ENSEMBLE_DIR / 'true_labels.npy', y_true)

print(f"✅ Predictions saved!")

## 10. Summary

In [ ]:
print("\n" + "="*80)
print("🎉 ENSEMBLE EVALUATION COMPLETE")
print("="*80)

print(f"\n📊 FINAL RESULTS:")
print(f"  Ensemble Accuracy: {ensemble_acc*100:.2f}%")
print(f"  Ensemble AUC:      {ensemble_auc:.4f}")
print(f"  Number of Models:  {len(models)}")

print(f"\n📈 INDIVIDUAL MODEL PERFORMANCE:")
for name, acc in individual_accuracies.items():
    print(f"  {name:<20} : {acc*100:.2f}%")

print(f"\n💾 SAVED FILES:")
print(f"  • {ENSEMBLE_DIR / 'ensemble_metrics.json'}")
print(f"  • {ENSEMBLE_DIR / 'ensemble_confusion_matrix.png'}")
print(f"  • {ENSEMBLE_DIR / 'model_comparison.png'}")
print(f"  • {ENSEMBLE_DIR / 'per_class_metrics.png'}")
print(f"  • {ENSEMBLE_DIR / 'ensemble_predictions.npy'}")

if ensemble_acc >= 0.95:
    print(f"\n✅ 🎯 PROJECT TARGET ACHIEVED!")
    print(f"   Ensemble accuracy {ensemble_acc*100:.2f}% exceeds 95% target")
else:
    print(f"\n⚠️ Target: {ensemble_acc*100:.2f}% < 95%")
    print(f"   Consider retraining models or adjusting ensemble strategy")

print("\n" + "="*80)